# 影片串流

[完整圖文教材](https://birdhackor.github.io/learn_to_yolo/lessons/18-video/)

先執行下一格的環境格，再閱讀、執行最後的完整實驗。各節互相獨立，不需要先跑其他節。

最後一格存著這段程式在 CPU 上實際執行的輸出；執行環境與其他實驗紀錄見網站的〈驗證範圍〉。範例不用預訓練權重，也不需要下載資料。

In [ ]:
# 全新 runtime 可由此格開始。只取得小型程式，不自動下載 LFS 資料。
import importlib.metadata
import os
from pathlib import Path
import subprocess
import sys

REF = 'lessons-v0.5.0'
base = Path("/content") if Path("/content").is_dir() else Path.cwd()
repository = base / ("learn_to_yolo_" + REF.replace(".", "_").replace("-", "_"))
if not repository.exists():
    environment = os.environ.copy()
    environment["GIT_LFS_SKIP_SMUDGE"] = "1"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF,
                    "https://github.com/birdhackor/learn_to_yolo.git", str(repository)],
                   env=environment, check=True)
actual_ref = subprocess.check_output(
    ["git", "-C", str(repository), "describe", "--tags", "--exact-match"], text=True
).strip()
if actual_ref != REF:
    raise RuntimeError("程式版本不同，請使用新的 runtime 或移除舊 clone 後重試。")

# 各節實驗都在 CPU 上執行，執行紀錄用的是 PyTorch 2.9.1。已經是 2.9.1 就沿用（CPU 或 CUDA 版都可以）；
# 其他版本會改裝成 2.9.1 的 CPU 版，之後這個工作階段就用不到 GPU，但各節實驗本來就不需要。
try:
    torch_version = importlib.metadata.version("torch").split("+")[0]
except importlib.metadata.PackageNotFoundError:
    torch_version = None
if torch_version != "2.9.1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.9.1",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    if "torch" in sys.modules:
        raise RuntimeError("PyTorch 已更新：請從選單重新啟動工作階段，再由第一格執行。")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                *['numpy==2.3.5', 'pillow==12.0.0', 'matplotlib==3.10.7']], check=True)
os.chdir(repository)
sys.path.insert(0, str(repository))
print("固定教材版本：", REF)
print("目前目錄：", repository)


## 本節可修改的完整實驗

先預測結果，再執行；確認輸出後，試做網頁的自主練習。

### 可選：真正的影片檔案與 tracking 接線

跑完主例後另開一個 code cell：

```python
!python -m pip install -r requirements-video.txt
!python scripts/verify_video_file.py
```

它自行產生 12 幀的無損 AVI，實際讀檔、模型推論、只替 class 0 配 track ID，並檢查 RGB 與框一致、影片資源有釋放；結果寫在 artifacts/runs/video-file/。沒有測實體相機。

In [1]:
"""Lazy RGB frame stream -> letterbox -> a locally trained detector -> overlays."""
import base64
import io
import json
import time
from dataclasses import dataclass
from pathlib import Path
import numpy as np
import torch
from PIL import Image, ImageDraw
from miniyolo.data import ShapeDataset, collate
from miniyolo.models import GridDetector
from miniyolo.targets import build_targets
from miniyolo.losses import grid_loss
from miniyolo.geometry import letterbox, undo_letterbox
from miniyolo.inference import decode_grid


@dataclass
class Frame:
    index: int
    timestamp_s: float
    rgb: np.ndarray  # uint8 HWC RGB; timestamp is source time, not processing latency


def synthetic_frames(count=12, fps=20):
    assert count >= 2 and fps > 0
    for i in range(count):
        rgb = np.full((64, 96, 3), 8, dtype=np.uint8)
        x = 4 + 4 * i
        # Clip at the right edge; once x >= width, this frame has no visible object.
        if x < rgb.shape[1]:
            rgb[20:34, x:min(x+14, rgb.shape[1])] = [242, 25, 25]
        yield Frame(i, i / fps, rgb)


def opencv_frames(source):
    """Optional real file/camera adapter. Never called by this lesson's main()."""
    import cv2  # optional: pip install opencv-python-headless
    capture = cv2.VideoCapture(source)
    try:
        if not capture.isOpened():
            raise RuntimeError(f'Cannot open video source: {source}')
        fps = capture.get(cv2.CAP_PROP_FPS)
        started, index = time.perf_counter(), 0
        while True:
            ok, bgr = capture.read()
            if not ok:
                break
            timestamp = index / fps if isinstance(source, str) and fps > 0 else time.perf_counter() - started
            yield Frame(index, timestamp, cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
            index += 1
    finally:
        capture.release()


def fit_detector():
    torch.manual_seed(7)
    dataset = ShapeDataset(n=32, size=64, seed=4100)
    images, targets = collate([dataset[i] for i in range(len(dataset))])
    encoded = build_targets(targets)
    model = GridDetector(width=8)
    optimizer = torch.optim.Adam(model.parameters(), lr=.01)
    for step in range(160):
        ids = torch.tensor([(step * 8 + j) % 32 for j in range(8)])
        optimizer.zero_grad()
        loss = grid_loss(model(images[ids]), {k: v[ids] for k, v in encoded.items()})['total']
        loss.backward()
        optimizer.step()
    return model.eval()


@torch.no_grad()
def run_stream(frames, model):
    """Consume any iterable of Frame, yield one result at a time; no camera required."""
    for frame in frames:
        assert frame.rgb.dtype == np.uint8 and frame.rgb.ndim == 3 and frame.rgb.shape[-1] == 3
        started = time.perf_counter()
        image = torch.from_numpy(frame.rgb.copy()).permute(2, 0, 1).float() / 255
        image, _, metadata = letterbox(image, torch.empty(0, 4), size=64)
        t1 = time.perf_counter()
        raw = model(image[None])
        t2 = time.perf_counter()
        # Score threshold .1, below the course's .25 display threshold (decode_grid's default):
        # on these letterboxed frames this 160-step detector's best score is under .25 in most
        # frames, so .1 also keeps its weaker detections. Box counts at .1 are not accuracy.
        prediction = decode_grid(raw, score_threshold=.1, nms_iou=.5)[0]
        prediction['boxes'] = undo_letterbox(prediction['boxes'], metadata)
        t3 = time.perf_counter()
        overlay = Image.fromarray(frame.rgb.copy())
        pen = ImageDraw.Draw(overlay)
        for box, score in zip(prediction['boxes'], prediction['scores']):
            pen.rectangle(tuple(float(v) for v in box), outline='yellow', width=1)
            pen.text((max(0, float(box[0])), max(0, float(box[1])-9)), f'{float(score):.2f}', fill='white')
        t4 = time.perf_counter()
        yield {'index': frame.index, 'source_timestamp_s': frame.timestamp_s, 'image': overlay, 'prediction': prediction,
               'ms': {'preprocess': (t1-started)*1000, 'model': (t2-t1)*1000,
                      'postprocess': (t3-t2)*1000, 'drawing': (t4-t3)*1000, 'total': (t4-started)*1000}}


def save_panel(results, fps, path):
    selected = [0, (len(results)-1)//2, len(results)-1]
    frames = '、'.join(str(i) for i in selected)
    parts = ['<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 960 290" role="img" aria-labelledby="t d">',
             f'<title id="t">移動畫面的實際偵測結果：{len(results)} 幀，來源每秒 {fps} 幀</title>',
             f'<desc id="d">三格依序是這次執行產生的第一、中間與最後一幀（第 {frames} 幀）的 RGB 畫面。'
             '黃框是模型的預測，框上的數字是 score；每格下方寫來源時間、這一幀的處理時間與框數。預測框不是真值。</desc>',
             '<rect width="960" height="290" fill="#0f172a"/>']
    for col, i in enumerate(selected):
        r = results[i]
        buffer = io.BytesIO()
        r['image'].save(buffer, format='PNG')
        data = base64.b64encode(buffer.getvalue()).decode()
        x = 15 + col * 315
        parts.append(f'<image x="{x}" y="20" width="288" height="192" href="data:image/png;base64,{data}"/>')
        parts.append(f'<text x="{x}" y="242" fill="white" font-family="sans-serif" font-size="21">第 {i} 幀：來源時間 {r["source_timestamp_s"]:.2f} 秒</text>')
        parts.append(f'<text x="{x}" y="272" fill="white" font-family="sans-serif" font-size="19">處理 {r["ms"]["total"]:.2f} ms｜框 {len(r["prediction"]["boxes"])} 個</text>')
    parts.append('</svg>')
    path.write_text('\n'.join(parts) + '\n')


def main(count=12, fps=20):
    torch.manual_seed(7)
    torch.set_num_threads(2)
    model = fit_detector()
    # Warm-up: push one throwaway copy of frame 0 through the whole pipeline and discard it,
    # so the one-time cost of each step's first call is not charged to frame 0. Its square
    # gets a box, so NMS and box/text drawing run too; list() is what runs the generator.
    warmup = list(run_stream([next(synthetic_frames(count=count, fps=fps))], model))
    assert len(warmup) == 1 and len(warmup[0]['prediction']['boxes']) > 0, 'warm-up must reach NMS and drawing'
    # Only this small GIF demo collects frames; run_stream itself remains lazy.
    results = list(run_stream(synthetic_frames(count=count, fps=fps), model))
    assert len(results) == count and all(r['image'].size == (96, 64) for r in results)
    assert [r['index'] for r in results] == list(range(count))
    assert abs(results[-1]['source_timestamp_s'] - (count-1)/fps) < 1e-8
    output = Path('artifacts/lesson-18')
    output.mkdir(parents=True, exist_ok=True)
    gif_duration_ms = round(1000/fps)
    results[0]['image'].save(output / 'stream.gif', save_all=True, append_images=[r['image'] for r in results[1:]], duration=gif_duration_ms, loop=0)
    # Median over all frames: the warm-up already paid the one-time setup, so frame 0 counts too.
    timing = {key: float(np.median([r['ms'][key] for r in results])) for key in results[0]['ms']}
    summary = {'frames': len(results), 'source_fps': fps, 'source_duration_s': count/fps,
               'source_last_timestamp_s': results[-1]['source_timestamp_s'], 'gif_frame_duration_ms': gif_duration_ms,
               'source_size_hw': [64, 96], 'object_visible_per_frame': [4+4*i < 96 for i in range(count)],
               'model_input_hw': [64, 64], 'boxes_per_frame': [len(r['prediction']['boxes']) for r in results],
               'warmup_frames': len(warmup), 'median_ms': timing, 'camera_adapter': 'provided, not executed',
               'limits': 'synthetic lazy producer; no capture/codec/display/network queue latency measured'}
    (output / 'report.json').write_text(json.dumps(summary, indent=2)+'\n')
    save_panel(results, fps, output / 'panel.svg')
    print(json.dumps(summary, indent=2))
    print('GIF: artifacts/lesson-18/stream.gif; actual static panel: artifacts/lesson-18/panel.svg')


if __name__ == '__main__':
    main()


{
  "frames": 12,
  "source_fps": 20,
  "source_duration_s": 0.6,
  "source_last_timestamp_s": 0.55,
  "gif_frame_duration_ms": 50,
  "source_size_hw": [
    64,
    96
  ],
  "object_visible_per_frame": [
    true,
    true,
    true,
    true,
    true,
    true,
    true,
    true,
    true,
    true,
    true,
    true
  ],
  "model_input_hw": [
    64,
    64
  ],
  "boxes_per_frame": [
    1,
    1,
    0,
    0,
    0,
    0,
    1,
    1,
    0,
    0,
    0,
    0
  ],
  "warmup_frames": 1,
  "median_ms": {
    "preprocess": 0.2764804958133027,
    "model": 0.30956500268075615,
    "postprocess": 0.43933201231993735,
    "drawing": 0.06989399844314903,
    "total": 1.1000390077242628
  },
  "camera_adapter": "provided, not executed",
  "limits": "synthetic lazy producer; no capture/codec/display/network queue latency measured"
}
GIF: artifacts/lesson-18/stream.gif; actual static panel: artifacts/lesson-18/panel.svg
